In [0]:
import time
import pandas as pd
from pyspark.storagelevel import StorageLevel
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql.types import DoubleType, StringType

CATALOG = "workspace"
SCHEMA = "default"
BASE_PATH = "/Volumes/workspace/default/analytics/optimizations_demo"

TABLES = {
    "customer": f"{CATALOG}.{SCHEMA}.saleslt_customer_demo",
    "product": f"{CATALOG}.{SCHEMA}.saleslt_product_demo",
    "header": f"{CATALOG}.{SCHEMA}.saleslt_sales_order_header_demo",
    "detail": f"{CATALOG}.{SCHEMA}.saleslt_sales_order_detail_demo",
}

PATHS = {
    "parquet_detail": f"{BASE_PATH}/detail_parquet",
    "partitioned_parquet_detail": f"{BASE_PATH}/detail_partitioned_parquet",
    "csv_detail": f"{BASE_PATH}/detail_csv",
    "delta_detail": f"{BASE_PATH}/detail_delta",
    "repartition_output": f"{BASE_PATH}/repartition_output",
    "coalesce_output": f"{BASE_PATH}/coalesce_output",
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.conf.set("spark.sql.shuffle.partitions", "200")


def force_action(result):
    if isinstance(result, DataFrame):
        return result.count()
    if isinstance(result, list):
        return len(result)
    if result is None:
        return "done"
    return result


def timed(label, func):
    start = time.perf_counter()
    result = func()
    metric = force_action(result)
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f}s | metric={metric}")
    return elapsed, metric


def compare(title, before_fn, after_fn, before_label="Without optimization", after_label="With optimization"):
    print(f"\n=== {title} ===")
    before_time, _ = timed(before_label, before_fn)
    after_time, _ = timed(after_label, after_fn)
    saved = before_time - after_time
    speedup = before_time / after_time if after_time else float("inf")
    print(f"Saved: {saved:.3f}s | Speedup: {speedup:.2f}x")
    return before_time, after_time


print("Schema ready:", f"{CATALOG}.{SCHEMA}")
print("Base path:", BASE_PATH)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Schema ready: workspace.default
Base path: /Volumes/workspace/default/analytics/optimizations_demo
Shuffle partitions: 200


In [0]:
n_customers = 50000
n_products = 5000
n_orders = 200000
n_details = 800000
base_date = F.to_date(F.lit("2023-01-01"))

customers = (
    spark.range(n_customers)
    .select(
        (F.col("id") + 1).alias("customer_id"),
        F.concat(F.lit("Customer "), F.lpad((F.col("id") + 1).cast("string"), 6, "0")).alias("customer_name"),
        F.concat(F.lit("customer"), (F.col("id") + 1).cast("string"), F.lit("@example.com")).alias("email_address"),
        F.when((F.col("id") % 4) == 0, "West")
         .when((F.col("id") % 4) == 1, "East")
         .when((F.col("id") % 4) == 2, "Central")
         .otherwise("South")
         .alias("region"),
        (((F.col("id") * 7) % 10) + 1).alias("territory_id"),
        F.date_sub(F.current_date(), (F.col("id") % 3650).cast("int")).alias("signup_date")
    )
)

products = (
    spark.range(n_products)
    .select(
        (F.col("id") + 1).alias("product_id"),
        F.concat(F.lit("Product "), F.lpad((F.col("id") + 1).cast("string"), 5, "0")).alias("product_name"),
        F.when((F.col("id") % 5) == 0, "Bikes")
         .when((F.col("id") % 5) == 1, "Components")
         .when((F.col("id") % 5) == 2, "Clothing")
         .when((F.col("id") % 5) == 3, "Accessories")
         .otherwise("Services")
         .alias("category"),
        F.when((F.col("id") % 3) == 0, "Standard")
         .when((F.col("id") % 3) == 1, "Premium")
         .otherwise("Economy")
         .alias("segment"),
        F.round(((F.col("id") % 900) + 50) * 1.15, 2).alias("list_price"),
        F.round(((F.col("id") % 700) + 25) * 0.82, 2).alias("standard_cost")
    )
)

headers = (
    spark.range(n_orders)
    .withColumn("sales_order_id", F.col("id") + 1)
    .withColumn("customer_id", ((F.col("id") * 17) % n_customers) + 1)
    .withColumn("territory_id", ((F.col("customer_id") * 3) % 10) + 1)
    .withColumn("order_date", F.date_add(base_date, (F.col("id") % 730).cast("int")))
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("status", F.when((F.col("id") % 15) == 0, "Cancelled").when((F.col("id") % 5) == 0, "Processing").otherwise("Shipped"))
    .withColumn("sub_total", F.round(((F.col("id") % 400) + 1) * 2.35, 2))
    .withColumn("tax_amt", F.round(F.col("sub_total") * 0.08, 2))
    .withColumn("freight", F.round(F.col("sub_total") * 0.03, 2))
    .withColumn("total_due", F.round(F.col("sub_total") + F.col("tax_amt") + F.col("freight"), 2))
    .drop("id")
)

details = (
    spark.range(n_details)
    .withColumn("sales_order_detail_id", F.col("id") + 1)
    .withColumn("sales_order_id", (F.col("id") % n_orders) + 1)
    .withColumn("product_id", ((F.col("id") * 13) % n_products) + 1)
    .withColumn("order_qty", ((F.col("id") % 5) + 1).cast("int"))
    .withColumn("unit_price", F.round((((F.col("product_id") % 500) + 10) * 1.15), 2))
    .withColumn("unit_price_discount", F.when(F.col("order_qty") >= 4, F.lit(0.10)).otherwise(F.lit(0.02)))
    .withColumn("line_amount", F.round(F.col("order_qty") * F.col("unit_price") * (1 - F.col("unit_price_discount")), 2))
    .withColumn("order_offset", ((F.col("sales_order_id") - 1) % 730).cast("int"))
    .withColumn("order_date", F.date_add(base_date, F.col("order_offset")))
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("customer_id", ((((F.col("sales_order_id") - 1) * 17) % n_customers) + 1))
    .drop("id", "order_offset")
)

customers.write.format("delta").mode("overwrite").saveAsTable(TABLES["customer"])
products.write.format("delta").mode("overwrite").saveAsTable(TABLES["product"])
headers.write.format("delta").mode("overwrite").partitionBy("order_year", "order_month").saveAsTable(TABLES["header"])
details.write.format("delta").mode("overwrite").partitionBy("order_year", "order_month").saveAsTable(TABLES["detail"])

details.drop("order_year", "order_month").write.mode("overwrite").parquet(PATHS["parquet_detail"])
details.write.mode("overwrite").partitionBy("order_year", "order_month").parquet(PATHS["partitioned_parquet_detail"])
details.limit(200000).write.mode("overwrite").option("header", True).csv(PATHS["csv_detail"])
details.write.format("delta").mode("overwrite").save(PATHS["delta_detail"])

print("Created tables:")
for name, table_name in TABLES.items():
    print(f"  {name}: {table_name}")
print("Created file paths:")
for name, path in PATHS.items():
    print(f"  {name}: {path}")

display(spark.table(TABLES["detail"]).limit(5))

display(spark.table(TABLES["detail"]).limit(5))

Created tables:
  customer: workspace.default.saleslt_customer_demo
  product: workspace.default.saleslt_product_demo
  header: workspace.default.saleslt_sales_order_header_demo
  detail: workspace.default.saleslt_sales_order_detail_demo
Created file paths:
  parquet_detail: /Volumes/workspace/default/analytics/optimizations_demo/detail_parquet
  partitioned_parquet_detail: /Volumes/workspace/default/analytics/optimizations_demo/detail_partitioned_parquet
  csv_detail: /Volumes/workspace/default/analytics/optimizations_demo/detail_csv
  delta_detail: /Volumes/workspace/default/analytics/optimizations_demo/detail_delta
  repartition_output: /Volumes/workspace/default/analytics/optimizations_demo/repartition_output
  coalesce_output: /Volumes/workspace/default/analytics/optimizations_demo/coalesce_output


sales_order_detail_id,sales_order_id,product_id,order_qty,unit_price,unit_price_discount,line_amount,order_date,order_year,order_month,customer_id
121,121,1561,1,81.65,0.02,80.02,2023-05-01,2023,5,2041
122,122,1574,2,96.6,0.02,189.34,2023-05-02,2023,5,2058
123,123,1587,3,111.55,0.02,327.96,2023-05-03,2023,5,2075
124,124,1600,4,126.5,0.1,455.4,2023-05-04,2023,5,2092
125,125,1613,5,141.45,0.1,636.53,2023-05-05,2023,5,2109


sales_order_detail_id,sales_order_id,product_id,order_qty,unit_price,unit_price_discount,line_amount,order_date,order_year,order_month,customer_id
121,121,1561,1,81.65,0.02,80.02,2023-05-01,2023,5,2041
122,122,1574,2,96.6,0.02,189.34,2023-05-02,2023,5,2058
123,123,1587,3,111.55,0.02,327.96,2023-05-03,2023,5,2075
124,124,1600,4,126.5,0.1,455.4,2023-05-04,2023,5,2092
125,125,1613,5,141.45,0.1,636.53,2023-05-05,2023,5,2109


In [0]:
parquet_df = spark.read.parquet(PATHS["parquet_detail"])

compare(
    "Predicate Pushdown",
    lambda: parquet_df.filter(F.year("order_date") == 2024).agg(F.sum("line_amount")),
    lambda: parquet_df.filter((F.col("order_date") >= F.lit("2024-01-01")) & (F.col("order_date") < F.lit("2025-01-01"))).agg(F.sum("line_amount")),
)

optimized_df = parquet_df.filter((F.col("order_date") >= F.lit("2024-01-01")) & (F.col("order_date") < F.lit("2025-01-01")))
optimized_df.explain("formatted")


=== Predicate Pushdown ===
Without optimization: 1.406s | metric=1
With optimization: 0.324s | metric=1
Saved: 1.081s | Speedup: 4.33x
== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [9]: [sales_order_detail_id#14677L, sales_order_id#14678L, product_id#14679L, order_qty#14680, unit_price#14681, unit_price_discount#14682, line_amount#14683, order_date#14684, customer_id#14685L]
DictionaryFilters: [((isnotnull(order_date#14684) AND (order_date#14684 >= 2024-01-01)) AND (order_date#14684 < 2025-01-01))]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/analytics/optimizations_demo/detail_parquet]
ReadSchema: struct<sales_order_detail_id:bigint,sales_order_id:bigint,product_id:bigint,order_qty:int,unit_price:double,unit_price_discount:double,line_amount:double,order_date:date,customer_id:bigint>
RequiredDataFilters: [isnotnull(order_date#14684), (order_date#14684 >= 2024-01-01), (order_date#1468

In [0]:
compare(
    "Column Pruning",
    lambda: spark.table(TABLES["detail"]).filter(F.col("order_year") == 2024).groupBy("product_id").agg(F.sum("line_amount")),
    lambda: spark.table(TABLES["detail"]).select("product_id", "line_amount", "order_year").filter(F.col("order_year") == 2024).groupBy("product_id").agg(F.sum("line_amount")),
)


=== Column Pruning ===
Without optimization: 0.911s | metric=5000
With optimization: 0.836s | metric=5000
Saved: 0.075s | Speedup: 1.09x


(0.9111157509998975, 0.835658882999951)

In [0]:
non_partitioned_df = spark.read.parquet(PATHS["parquet_detail"])
partitioned_df = spark.table(TABLES["detail"])

compare(
    "Partition Pruning",
    lambda: non_partitioned_df.filter((F.col("order_date") >= F.lit("2024-06-01")) & (F.col("order_date") < F.lit("2024-07-01"))).agg(F.sum("line_amount")),
    lambda: partitioned_df.filter((F.col("order_year") == 2024) & (F.col("order_month") == 6)).agg(F.sum("line_amount")),
)

partitioned_df.filter((F.col("order_year") == 2024) & (F.col("order_month") == 6)).explain("formatted")


=== Partition Pruning ===
Without optimization: 1.091s | metric=1
With optimization: 0.638s | metric=1
Saved: 0.453s | Speedup: 1.71x
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo (1)


(1) PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo
Output [11]: [sales_order_detail_id#14875L, sales_order_id#14876L, product_id#14877L, order_qty#14878, unit_price#14879, unit_price_discount#14880, line_amount#14881, order_date#14882, customer_id#14885L, order_year#14883, order_month#14884]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-tafcp/uc/d75358e5-0aa5-42e4-bde9-7edd4b246667/5ece82ed-b8e8-4702-ad55-3c8073d29aa0/__unitystorage/catalogs/b3b71af4-4f22-4207-8fc9-3d7a06640316/tables/fdbb5e13-fe44-492c-a5ad-f2549bf96708]
PartitionFilters: [isnotnull(order_year#14883), isnotnull(order_month#14884), (order_year#14883 = 2024), (order_month#14884 = 6)]


In [0]:
partitioned_parquet_df = spark.read.parquet(PATHS["partitioned_parquet_detail"])
plain_parquet_df = spark.read.parquet(PATHS["parquet_detail"])

compare(
    "Proper Data Partitioning",
    lambda: plain_parquet_df.filter((F.col("order_date") >= F.lit("2024-03-01")) & (F.col("order_date") < F.lit("2024-04-01"))).groupBy("customer_id").agg(F.sum("line_amount")),
    lambda: partitioned_parquet_df.filter((F.col("order_year") == 2024) & (F.col("order_month") == 3)).groupBy("customer_id").agg(F.sum("line_amount")),
)


=== Proper Data Partitioning ===
Without optimization: 1.475s | metric=5617
With optimization: 5.244s | metric=5617
Saved: -3.769s | Speedup: 0.28x


(1.4746232879999752, 5.243610664000016)

In [0]:
detail_2024 = spark.table(TABLES["detail"]).filter(F.col("order_year") == 2024)

compare(
    "Repartition Optimization",
    lambda: detail_2024.repartition(200).groupBy("customer_id").agg(F.sum("line_amount")),
    lambda: detail_2024.repartition(32, "customer_id").groupBy("customer_id").agg(F.sum("line_amount")),
)


=== Repartition Optimization ===
Without optimization: 1.251s | metric=50000
With optimization: 0.729s | metric=50000
Saved: 0.522s | Speedup: 1.72x


(1.2507418999998663, 0.7290859489999093)

In [0]:
small_df = spark.table(TABLES["detail"]).filter((F.col("order_year") == 2024) & (F.col("order_month") == 1))

compare(
    "Coalesce Optimization",
    lambda: small_df.repartition(32).write.mode("overwrite").parquet(PATHS["repartition_output"]),
    lambda: small_df.coalesce(8).write.mode("overwrite").parquet(PATHS["coalesce_output"]),
)


=== Coalesce Optimization ===
Without optimization: 2.424s | metric=done
With optimization: 1.568s | metric=done
Saved: 0.855s | Speedup: 1.55x


(2.423557597999661, 1.5682714619997569)

In [0]:
detail_df = spark.table(TABLES["detail"]).select("sales_order_id", "product_id", "line_amount")
product_df = spark.table(TABLES["product"]).select("product_id", "category", "segment")

compare(
    "Broadcast Joins",
    lambda: detail_df.join(product_df, "product_id").groupBy("category").agg(F.sum("line_amount")),
    lambda: detail_df.join(F.broadcast(product_df), "product_id").groupBy("category").agg(F.sum("line_amount")),
)


=== Broadcast Joins ===
Without optimization: 1.424s | metric=5
With optimization: 1.044s | metric=5
Saved: 0.380s | Speedup: 1.36x


(1.4238544780000666, 1.0443322870000884)

In [0]:
print("Bucketing example:")
print("On Databricks with Unity Catalog and Delta, the preferred physical layout features are partitioning, OPTIMIZE, and Z-ORDER rather than maintaining bucketed tables from notebook code.")
print("Use the later OPTIMIZE and Z-ORDER cells as the Databricks-native replacement for most bucketing scenarios.")

Bucketing example:
On Databricks with Unity Catalog and Delta, the preferred physical layout features are partitioning, OPTIMIZE, and Z-ORDER rather than maintaining bucketed tables from notebook code.
Use the later OPTIMIZE and Z-ORDER cells as the Databricks-native replacement for most bucketing scenarios.


In [0]:
salt_buckets = 8
skew_fact = (
    spark.range(300000)
    .select(
        F.when(F.col("id") < 220000, F.lit(1)).otherwise((F.col("id") % 5000) + 2).alias("customer_id"),
        F.round((F.col("id") % 1000) * 1.25, 2).alias("amount")
    )
)

skew_dim = spark.range(5001).select((F.col("id") + 1).alias("customer_id"), ((F.col("id") % 10) + 1).alias("region_id"))

salted_fact = skew_fact.withColumn("salt", F.when(F.col("customer_id") == 1, (F.rand(7) * salt_buckets).cast("int")).otherwise(F.lit(0)))
hot_dim = skew_dim.filter(F.col("customer_id") == 1).select("customer_id", "region_id", F.explode(F.array(*[F.lit(i) for i in range(salt_buckets)])).alias("salt"))
cold_dim = skew_dim.filter(F.col("customer_id") != 1).select("customer_id", "region_id", F.lit(0).alias("salt"))
salted_dim = hot_dim.unionByName(cold_dim)

In [0]:
print("AQE is managed automatically on Databricks serverless compute.")
compare(
    "Adaptive Query Execution (AQE Friendly)",
    lambda: spark.table(TABLES["detail"]).repartition(200).groupBy("customer_id").agg(F.sum("line_amount")),
    lambda: spark.table(TABLES["detail"]).groupBy("customer_id").agg(F.sum("line_amount")),
)

spark.table(TABLES["detail"]).groupBy("customer_id").agg(F.sum("line_amount")).explain("formatted")

AQE is managed automatically on Databricks serverless compute.

=== Adaptive Query Execution (AQE Friendly) ===
Without optimization: 1.327s | metric=50000
With optimization: 0.627s | metric=50000
Saved: 0.700s | Speedup: 2.12x
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo (1)


(1) PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo
Output [4]: [line_amount#15515, customer_id#15519L, order_year#15517, order_month#15518]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-tafcp/uc/d75358e5-0aa5-42e4-bde9-7edd4b246667/5ece82ed-b8e8-4702-ad55-3c8073d29aa0/__

In [0]:
filtered_headers = spark.table(TABLES["header"]).filter((F.col("order_year") == 2024) & (F.col("customer_id") <= 2000)).select("sales_order_id")

compare(
    "Dynamic Partition Pruning",
    lambda: spark.read.parquet(PATHS["parquet_detail"]).join(filtered_headers, "sales_order_id").agg(F.sum("line_amount")),
    lambda: spark.table(TABLES["detail"]).join(filtered_headers, "sales_order_id").agg(F.sum("line_amount")),
)

spark.table(TABLES["detail"]).join(filtered_headers, "sales_order_id").explain("formatted")


=== Dynamic Partition Pruning ===
Without optimization: 1.196s | metric=1
With optimization: 0.456s | metric=1
Saved: 0.740s | Speedup: 2.62x
== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonBroadcastHashJoin Inner (7)
            :- PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo (1)
            +- PhotonShuffleExchangeSource (6)
               +- PhotonShuffleMapStage (5)
                  +- PhotonShuffleExchangeSink (4)
                     +- PhotonProject (3)
                        +- PhotonScan parquet workspace.default.saleslt_sales_order_header_demo (2)


(1) PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo
Output [11]: [sales_order_detail_id#15672L, sales_order_id#15673L, product_id#15674L, order_qty#15675, unit_price#15676, unit_price_discount#15677, line_amount#15678, order_date#15679, customer_id#15682L, order_year#

In [0]:
print("Cache() Optimization:")
print("NOTE: .cache() and .persist() are not supported on Databricks serverless compute.")
print("Serverless compute automatically manages caching and memory optimization.")
print("On traditional clusters, you would use .cache() to reuse DataFrames across multiple actions.")
print("Example pattern (for non-serverless clusters):")
print("  df.cache().count()  # Materialize the cache")
print("  df.groupBy(...).agg(...)  # Reuses cached data")
print("  df.unpersist()  # Release cache when done")

Cache() Optimization:
NOTE: .cache() and .persist() are not supported on Databricks serverless compute.
Serverless compute automatically manages caching and memory optimization.
On traditional clusters, you would use .cache() to reuse DataFrames across multiple actions.
Example pattern (for non-serverless clusters):
  df.cache().count()  # Materialize the cache
  df.groupBy(...).agg(...)  # Reuses cached data
  df.unpersist()  # Release cache when done


In [0]:
print("Cache() Optimization:")
print("NOTE: .cache() and .persist() are not supported on Databricks serverless compute.")
print("Serverless compute automatically manages caching and memory optimization.")
print("On traditional clusters, you would use .cache() to reuse DataFrames across multiple actions.")
print("Example pattern (for non-serverless clusters):")
print("  df.cache().count()  # Materialize the cache")
print("  df.groupBy(...).agg(...)  # Reuses cached data")
print("  df.unpersist()  # Release cache when done")

Cache() Optimization:
NOTE: .cache() and .persist() are not supported on Databricks serverless compute.
Serverless compute automatically manages caching and memory optimization.
On traditional clusters, you would use .cache() to reuse DataFrames across multiple actions.
Example pattern (for non-serverless clusters):
  df.cache().count()  # Materialize the cache
  df.groupBy(...).agg(...)  # Reuses cached data
  df.unpersist()  # Release cache when done


In [0]:
print("Persist() Optimization:")
print("NOTE: .persist() with custom StorageLevel is not supported on Databricks serverless compute.")
print("Serverless compute automatically manages memory and disk caching.")
print("On traditional clusters, you would use .persist(StorageLevel) to control caching strategy.")
print("Example pattern (for non-serverless clusters):")
print("  df.persist(StorageLevel.MEMORY_AND_DISK).count()  # Materialize with specific storage")
print("  df.groupBy(...).agg(...)  # Reuses persisted data")
print("  df.unpersist()  # Release when done")

Persist() Optimization:
NOTE: .persist() with custom StorageLevel is not supported on Databricks serverless compute.
Serverless compute automatically manages memory and disk caching.
On traditional clusters, you would use .persist(StorageLevel) to control caching strategy.
Example pattern (for non-serverless clusters):
  df.persist(StorageLevel.MEMORY_AND_DISK).count()  # Materialize with specific storage
  df.groupBy(...).agg(...)  # Reuses persisted data
  df.unpersist()  # Release when done


In [0]:
print("Kryo Serialization")
print("On Databricks serverless, serializer configuration is managed by the platform and not typically toggled from notebook code.")
compare(
    "Kryo Serialization Approximation",
    lambda: spark.table(TABLES["detail"]).selectExpr("CAST(customer_id AS STRING) AS customer_id", "line_amount").groupBy("customer_id").agg(F.sum("line_amount")),
    lambda: spark.table(TABLES["detail"]).select("customer_id", "line_amount").groupBy("customer_id").agg(F.sum("line_amount")),
)
print("The optimized path keeps a compact native numeric type, which approximates the benefit of efficient serialization.")

Kryo Serialization
On Databricks serverless, serializer configuration is managed by the platform and not typically toggled from notebook code.

=== Kryo Serialization Approximation ===
Without optimization: 0.674s | metric=50000
With optimization: 0.540s | metric=50000
Saved: 0.134s | Speedup: 1.25x
The optimized path keeps a compact native numeric type, which approximates the benefit of efficient serialization.


In [0]:
print("Tungsten Execution Engine")
compare(
    "Tungsten Friendly Expressions",
    lambda: spark.table(TABLES["detail"]).select("sales_order_id", "line_amount").groupBy("sales_order_id").agg(F.sum("line_amount")),
    lambda: spark.table(TABLES["detail"]).selectExpr("sales_order_id", "CAST(line_amount AS DOUBLE) AS line_amount").groupBy("sales_order_id").agg(F.sum("line_amount")),
)
print("Databricks uses Tungsten-style whole-stage native execution automatically for Spark SQL expressions.")

Tungsten Execution Engine

=== Tungsten Friendly Expressions ===
Without optimization: 0.519s | metric=200000
With optimization: 0.541s | metric=200000
Saved: -0.022s | Speedup: 0.96x
Databricks uses Tungsten-style whole-stage native execution automatically for Spark SQL expressions.


In [0]:
print("Whole-Stage Code Generation")
compare(
    "Whole-Stage Code Generation Friendly Plan",
    lambda: spark.table(TABLES["detail"]).select("customer_id", "order_qty", "line_amount").filter(F.col("order_qty") >= 2).groupBy("customer_id").agg(F.sum("line_amount")),
    lambda: spark.table(TABLES["detail"]).select("customer_id", "order_qty", "line_amount").filter(F.col("order_qty") >= 2).withColumn("rounded_amount", F.round("line_amount", 2)).groupBy("customer_id").agg(F.sum("rounded_amount")),
)
spark.table(TABLES["detail"]).select("customer_id", "order_qty", "line_amount").filter(F.col("order_qty") >= 2).groupBy("customer_id").agg(F.sum("line_amount")).explain("formatted")

Whole-Stage Code Generation

=== Whole-Stage Code Generation Friendly Plan ===
Without optimization: 0.706s | metric=40000
With optimization: 0.634s | metric=40000
Saved: 0.072s | Speedup: 1.11x
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo (1)


(1) PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo
Output [5]: [order_qty#16808, line_amount#16811, customer_id#16815L, order_year#16813, order_month#16814]
DictionaryFilters: [(order_qty#16808 >= 2)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-tafcp/uc/d75358e5-0aa5-42e4-bde9-7edd4b246667/5ece82ed-b8

In [0]:
large_df = spark.table(TABLES["detail"]).select("sales_order_id", "product_id", "line_amount")

compare(
    "Avoid collect() on Large Data",
    lambda: large_df.limit(50000).collect(),
    lambda: large_df.limit(50000).toLocalIterator(),
)
print("Use distributed writes, aggregations, or limited iteration instead of bringing large datasets to the driver.")


=== Avoid collect() on Large Data ===
Without optimization: 0.998s | metric=50000
With optimization: 0.000s | metric=<generator object DataFrame.toLocalIterator at 0xff3edc60ef00>
Saved: 0.998s | Speedup: 18708.31x
Use distributed writes, aggregations, or limited iteration instead of bringing large datasets to the driver.


In [0]:
@udf(returnType=StringType())
def discount_band_udf(amount):
    if amount is None:
        return None
    if amount >= 1000:
        return "High"
    if amount >= 300:
        return "Medium"
    return "Low"

compare(
    "Avoid UDFs (Use Built-in Functions)",
    lambda: spark.table(TABLES["detail"]).withColumn("band", discount_band_udf("line_amount")).groupBy("band").count(),
    lambda: spark.table(TABLES["detail"]).withColumn(
        "band",
        F.when(F.col("line_amount") >= 1000, "High").when(F.col("line_amount") >= 300, "Medium").otherwise("Low")
    ).groupBy("band").count(),
)


=== Avoid UDFs (Use Built-in Functions) ===
Without optimization: 14.748s | metric=3
With optimization: 0.619s | metric=3
Saved: 14.129s | Speedup: 23.84x


(14.74757919600006, 0.6186941229998411)

In [0]:
@pandas_udf(DoubleType())
def discounted_amount_pandas(amount: pd.Series) -> pd.Series:
    return amount * 0.95

@udf(returnType=DoubleType())
def discounted_amount_udf(amount):
    if amount is None:
        return None
    return float(amount) * 0.95

compare(
    "Pandas UDFs Instead of Traditional UDFs",
    lambda: spark.table(TABLES["detail"]).withColumn("discounted", discounted_amount_udf("line_amount")).agg(F.sum("discounted")),
    lambda: spark.table(TABLES["detail"]).withColumn("discounted", discounted_amount_pandas("line_amount")).agg(F.sum("discounted")),
)


=== Pandas UDFs Instead of Traditional UDFs ===
Without optimization: 0.592s | metric=1
With optimization: 0.539s | metric=1
Saved: 0.053s | Speedup: 1.10x


(0.5923216320002211, 0.5393034899998383)

In [0]:
joined_df = spark.table(TABLES["detail"]).join(spark.table(TABLES["product"]), "product_id")

compare(
    "Filter Early, Filter Often",
    lambda: joined_df.groupBy("category").agg(F.sum("line_amount")).filter(F.col("category") == "Bikes"),
    lambda: spark.table(TABLES["detail"]).filter(F.col("order_year") == 2024).join(
        spark.table(TABLES["product"]).filter(F.col("category") == "Bikes"),
        "product_id"
    ).groupBy("category").agg(F.sum("line_amount")),
)


=== Filter Early, Filter Often ===
Without optimization: 0.864s | metric=1
With optimization: 0.884s | metric=1
Saved: -0.021s | Speedup: 0.98x


(0.8635050069997305, 0.8840350090003994)

In [0]:
detail_df = spark.table(TABLES["detail"]).alias("detail")
header_df = spark.table(TABLES["header"]).alias("header")
customer_df = spark.table(TABLES["customer"]).alias("customer")

wide_join = detail_df.join(header_df, "sales_order_id").join(customer_df, "customer_id")

compare(
    "Select Required Columns Only",
    lambda: wide_join.filter(F.col("detail.order_year") == 2024).groupBy("region").agg(F.sum("line_amount")),
    lambda: wide_join.select("region", "line_amount", "detail.order_year").filter(F.col("detail.order_year") == 2024).groupBy("region").agg(F.sum("line_amount")),
)


=== Select Required Columns Only ===
Without optimization: 1.424s | metric=4
With optimization: 1.303s | metric=4
Saved: 0.121s | Speedup: 1.09x


(1.4235845460002565, 1.3025748119998752)

In [0]:
detail_base = spark.table(TABLES["detail"]).filter(F.col("order_year") == 2024)
original_shuffle = spark.conf.get("spark.sql.shuffle.partitions")

print("\n=== Optimize Shuffle Partitions (spark.sql.shuffle.partitions) ===")
spark.conf.set("spark.sql.shuffle.partitions", "200")
before_time, _ = timed("Without optimization", lambda: detail_base.groupBy("customer_id").agg(F.sum("line_amount")))
spark.conf.set("spark.sql.shuffle.partitions", "32")
after_time, _ = timed("With optimization", lambda: detail_base.groupBy("customer_id").agg(F.sum("line_amount")))
spark.conf.set("spark.sql.shuffle.partitions", original_shuffle)
print(f"Saved: {before_time - after_time:.3f}s | Speedup: {before_time / after_time if after_time else float('inf'):.2f}x")


=== Optimize Shuffle Partitions (spark.sql.shuffle.partitions) ===
Without optimization: 0.657s | metric=50000
With optimization: 0.534s | metric=50000
Saved: 0.123s | Speedup: 1.23x


In [0]:
skewed_group_df = (
    spark.range(400000)
    .select(
        F.when(F.col("id") < 300000, F.lit(1)).otherwise((F.col("id") % 1000) + 2).alias("skew_key"),
        (F.col("id") % 100).cast("double").alias("amount")
    )
)

balanced_group_df = skewed_group_df.withColumn("salt", F.when(F.col("skew_key") == 1, (F.rand(11) * 8).cast("int")).otherwise(F.lit(0)))

compare(
    "Data Skew Handling",
    lambda: skewed_group_df.groupBy("skew_key").agg(F.sum("amount")),
    lambda: balanced_group_df.groupBy("skew_key", "salt").agg(F.sum("amount").alias("partial_amount")).groupBy("skew_key").agg(F.sum("partial_amount")),
)


=== Data Skew Handling ===
Without optimization: 0.256s | metric=1001
With optimization: 0.316s | metric=1001
Saved: -0.060s | Speedup: 0.81x


(0.2555200150000019, 0.31569911499991576)

In [0]:
headers_small = spark.table(TABLES["header"]).filter(F.col("order_year") == 2024)
details_all = spark.table(TABLES["detail"])
products_small = spark.table(TABLES["product"]).filter(F.col("category") == "Bikes")

compare(
    "Join Order Optimization",
    lambda: details_all.join(spark.table(TABLES["product"]), "product_id").join(headers_small, "sales_order_id").groupBy("category").agg(F.sum("line_amount")),
    lambda: headers_small.join(details_all, "sales_order_id").join(F.broadcast(products_small), "product_id").groupBy("category").agg(F.sum("line_amount")),
)


=== Join Order Optimization ===
Without optimization: 1.177s | metric=5
With optimization: 1.248s | metric=1
Saved: -0.071s | Speedup: 0.94x


(1.17739215399979, 1.2481312529998831)

In [0]:
compare(
    "Use Parquet Format",
    lambda: spark.read.option("header", True).csv(PATHS["csv_detail"]).filter(F.col("order_qty") >= 3).groupBy("product_id").count(),
    lambda: spark.read.parquet(PATHS["parquet_detail"]).filter(F.col("order_qty") >= 3).groupBy("product_id").count(),
)


=== Use Parquet Format ===
Without optimization: 2.814s | metric=3000
With optimization: 1.099s | metric=3000
Saved: 1.715s | Speedup: 2.56x


(2.813866304000385, 1.0988429330000145)

In [0]:
compare(
    "Use Delta Lake Format",
    lambda: spark.read.parquet(PATHS["parquet_detail"]).filter((F.col("order_date") >= F.lit("2024-01-01")) & (F.col("order_date") < F.lit("2025-01-01"))).groupBy("customer_id").agg(F.sum("line_amount")),
    lambda: spark.read.format("delta").load(PATHS["delta_detail"]).filter(F.col("order_year") == 2024).groupBy("customer_id").agg(F.sum("line_amount")),
)


=== Use Delta Lake Format ===
Without optimization: 0.967s | metric=50000
With optimization: 0.766s | metric=50000
Saved: 0.201s | Speedup: 1.26x


(0.966550503000235, 0.7656279540001378)

In [0]:
spark.sql(f"OPTIMIZE {TABLES['detail']} ZORDER BY (customer_id, product_id)")
print("Ran Z-ORDER on", TABLES["detail"])
compare(
    "Z-Ordering (Delta Lake)",
    lambda: spark.read.format("delta").load(PATHS["delta_detail"]).filter((F.col("customer_id") <= 1000) & (F.col("product_id") <= 100)).agg(F.sum("line_amount")),
    lambda: spark.table(TABLES["detail"]).filter((F.col("customer_id") <= 1000) & (F.col("product_id") <= 100)).agg(F.sum("line_amount")),
)

Ran Z-ORDER on workspace.default.saleslt_sales_order_detail_demo

=== Z-Ordering (Delta Lake) ===
Without optimization: 0.794s | metric=1
With optimization: 0.379s | metric=1
Saved: 0.415s | Speedup: 2.10x


(0.7935520559999532, 0.3785702149998542)

In [0]:
spark.sql(f"OPTIMIZE {TABLES['header']}")
print("Ran OPTIMIZE on", TABLES["header"])
spark.sql(f"DESCRIBE DETAIL {TABLES['header']}").show(truncate=False)

Ran OPTIMIZE on workspace.default.saleslt_sales_order_header_demo
+------+------------------------------------+-------------------------------------------------+-----------+--------+-----------------------+-------------------+-------------------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                                             |description|location|createdAt              |lastModified       |partitionColumns         |clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                   

In [0]:
spark.sql(f"VACUUM {TABLES['detail']} RETAIN 168 HOURS")
print("Ran VACUUM on", TABLES["detail"])

spark.sql(f"DESCRIBE HISTORY {TABLES['detail']}").show(truncate=False)

Ran VACUUM on workspace.default.saleslt_sales_order_detail_demo
+-------+-------------------+--------------+--------------------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName            |operation                        |operationParameters                          

In [0]:
plan_df = spark.table(TABLES["detail"]).filter((F.col("order_year") == 2024) & (F.col("order_month") == 6)).join(
    F.broadcast(spark.table(TABLES["product"]).filter(F.col("category") == "Bikes")),
    "product_id"
).groupBy("product_id").agg(F.sum("line_amount").alias("sales_amount"))

print("Explain plan:")
plan_df.explain("formatted")

compare(
    "Explain Plan Analysis Timing",
    lambda: spark.table(TABLES["detail"]).join(spark.table(TABLES["product"]), "product_id").groupBy("product_id").agg(F.sum("line_amount")),
    lambda: plan_df,
)

Explain plan:
== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonProject (16)
         +- PhotonGroupingAgg (15)
            +- PhotonShuffleExchangeSource (14)
               +- PhotonShuffleMapStage (13)
                  +- PhotonShuffleExchangeSink (12)
                     +- PhotonGroupingAgg (11)
                        +- PhotonProject (10)
                           +- PhotonBroadcastHashJoin Inner (9)
                              :- PhotonGroupingAgg (3)
                              :  +- PhotonProject (2)
                              :     +- PhotonScan parquet workspace.default.saleslt_sales_order_detail_demo (1)
                              +- PhotonShuffleExchangeSource (8)
                                 +- PhotonShuffleMapStage (7)
                                    +- PhotonShuffleExchangeSink (6)
                                       +- PhotonProject (5)
                      

(0.7522587499997826, 1.0602693420000833)